# T26. Cobertura parcial de conjuntos

**Objetivo.** Minimizar estaciones para cubrir en ambos extremos al menos el 80 % de los viajes elegibles del ajuste. Se prueban los cuatro radios del plan sobre T24.

Cada estación tiene una variable binaria. Un grupo de viajes puede contar en el objetivo de cobertura si hay al menos una estación abierta en su conjunto de orígenes alcanzables y una en su conjunto de destinos. Se agrupan viajes que tienen los mismos conjuntos y se conserva su multiplicidad. La variable continua de cada grupo está entre cero y uno. Con las aperturas binarias, el recuento de cobertura factible se verifica después por viaje.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from redes import CoberturaOD, OBJETIVO_PCT, Problema, RADIOS, guardar

LIMITE_S = 10
p = Problema.del_ajuste()
resumen, redes = [], []
for radio in RADIOS:
    modelo = CoberturaOD(p, radio)
    prefijo, cobertura = modelo.voraz()
    factibles = np.flatnonzero(cobertura >= p.objetivo)
    if not len(factibles):
        raise ValueError(f"El objetivo de {OBJETIVO_PCT} % es inviable con {radio} m incluso con todo el catálogo.")
    k = int(factibles[0]) + 1
    semilla = prefijo[:k]
    for nuevo_k in range(k - 1, max(k - 9, 0), -1):
        tentativa = modelo.intercambiar(prefijo[:nuevo_k])
        if modelo.cubiertos(tentativa) >= p.objetivo:
            semilla = tentativa
    elegida, solver = modelo.resolver(semilla, LIMITE_S, objetivo=p.objetivo)
    red_id = f"lscp_r{radio}_k{len(elegida)}"
    medidas = p.medir(elegida, radio)
    assert medidas["viajes_cubiertos"] >= p.objetivo
    resumen.append({"network_id": red_id, "metodo": "lscp", "radio_m": radio, "k": len(elegida),
                    "objetivo_pct": OBJETIVO_PCT, "objetivo_viajes": p.objetivo, "k_voraz": k,
                    "grupos_od": len(modelo.pesos), **medidas, **solver})
    redes.append(p.estaciones(elegida, red_id))

resumen = pd.DataFrame(resumen)
guardar(resumen, "T26_resultados")
guardar(pd.concat(redes, ignore_index=True), "T26_redes")
display(resumen[["radio_m", "k", "k_voraz", "pct_viajes_cubiertos", "cota_obj",
                 "brecha_pct", "origen_solucion", "solver_s"]].round(3))

,radio_m,k,k_voraz,pct_viajes_cubiertos,cota_obj,brecha_pct,origen_solucion,solver_s
0,200,289,293,80.108,289,0.000,MILP,2.415
1,250,225,228,80.036,224,0.444,heuristica OD,10.109
2,300,159,163,80.030,62,61.006,heuristica OD,11.806
3,400,86,87,80.862,33,61.628,heuristica OD,12.431


**Lectura.** La cota de LSCP se expresa en estaciones. Si el solver alcanza su límite, se conserva la mejor solución factible contrastada sobre los viajes. La coincidencia entre cota y número de estaciones certifica la optimalidad.

**Auditoría de la cota.** Se mantienen los emplazamientos de la solución publicada. La relajación compacta de `certificados.py` representa una sola vez cada conjunto de candidatos alcanzables y enlaza cada grupo OD con sus dos conjuntos. Con aperturas binarias es equivalente al modelo de cobertura OD. Con aperturas fraccionarias, compartir la variable de acceso evita repetir la misma suma. La relajación conserva la validez de la cota; la mejora procede de resolver el problema lineal compacto y verificar su dual. Los multiplicadores se escalan a enteros y se penalizan los residuos para que el redondeo no invalide la cota. `T26_certificados.csv` publica por radio el intervalo entero, el margen en viajes sobre el objetivo y la comparación con la heurística voraz. Para 300 m, el mínimo queda entre 154 y 159 estaciones. Las 159 estaciones cubren 13 373 viajes, cinco por encima de los 13 368 necesarios.